In [3]:
!pip install timm -q

import os
import torch
import timm
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, classification_report

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/kaggle/input/datasets/mohitkarthiekeya/final-data/Final_Data"
print("Using device:", device)

Using device: cuda


In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

In [6]:
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=val_transform)

class_names = train_dataset.classes
print("Classes:", class_names)

# 🔥 Handle imbalance
targets = [label for _, label in train_dataset]
class_counts = np.bincount(targets)

class_weights = 1. / class_counts
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

Classes: ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']


In [7]:
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [8]:
model_eff = timm.create_model('tf_efficientnet_b4', pretrained=True, num_classes=4).to(device)
model_inc = timm.create_model('inception_v3', pretrained=True, num_classes=4).to(device)

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/95.5M [00:00<?, ?B/s]

In [9]:

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_eff = optim.AdamW(model_eff.parameters(), lr=3e-4)
optimizer_inc = optim.AdamW(model_inc.parameters(), lr=3e-4)

scheduler_eff = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff, T_max=30)
scheduler_inc = optim.lr_scheduler.CosineAnnealingLR(optimizer_inc, T_max=30)

In [10]:
def train_effnet(epochs=30):
    best_f1 = 0

    for epoch in range(epochs):
        model_eff.train()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer_eff.zero_grad()
            outputs = model_eff(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer_eff.step()

        scheduler_eff.step()

        # VALIDATION
        model_eff.eval()
        preds, gts = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)

                out = model_eff(images)
                pred = torch.argmax(out, dim=1)

                preds.extend(pred.cpu().numpy())
                gts.extend(labels.numpy())

        f1 = f1_score(gts, preds, average='macro')
        print(f"EffNet Epoch {epoch+1} | F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model_eff.state_dict(), "best_eff.pth")

    print("✅ EfficientNet Done")

train_effnet()

EffNet Epoch 1 | F1: 0.8751
EffNet Epoch 2 | F1: 0.8886
EffNet Epoch 3 | F1: 0.8965
EffNet Epoch 4 | F1: 0.9037
EffNet Epoch 5 | F1: 0.9282
EffNet Epoch 6 | F1: 0.9240
EffNet Epoch 7 | F1: 0.9264
EffNet Epoch 8 | F1: 0.9330
EffNet Epoch 9 | F1: 0.9316
EffNet Epoch 10 | F1: 0.9206
EffNet Epoch 11 | F1: 0.9283
EffNet Epoch 12 | F1: 0.9312
EffNet Epoch 13 | F1: 0.9358
EffNet Epoch 14 | F1: 0.9356
EffNet Epoch 15 | F1: 0.9407
EffNet Epoch 16 | F1: 0.9328
EffNet Epoch 17 | F1: 0.9481
EffNet Epoch 18 | F1: 0.9401
EffNet Epoch 19 | F1: 0.9417
EffNet Epoch 20 | F1: 0.9410
EffNet Epoch 21 | F1: 0.9519
EffNet Epoch 22 | F1: 0.9516
EffNet Epoch 23 | F1: 0.9529
EffNet Epoch 24 | F1: 0.9479
EffNet Epoch 25 | F1: 0.9460
EffNet Epoch 26 | F1: 0.9542
EffNet Epoch 27 | F1: 0.9529
EffNet Epoch 28 | F1: 0.9492
EffNet Epoch 29 | F1: 0.9529
EffNet Epoch 30 | F1: 0.9522
✅ EfficientNet Done


In [11]:
def train_inception(epochs=30):
    best_f1 = 0

    for epoch in range(epochs):
        model_inc.train()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # 🔥 resize ONLY here
            images_299 = F.interpolate(images, size=(299,299), mode='bilinear')

            optimizer_inc.zero_grad()
            outputs = model_inc(images_299)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer_inc.step()

        scheduler_inc.step()

        # VALIDATION
        model_inc.eval()
        preds, gts = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                images_299 = F.interpolate(images, size=(299,299), mode='bilinear')

                out = model_inc(images_299)
                pred = torch.argmax(out, dim=1)

                preds.extend(pred.cpu().numpy())
                gts.extend(labels.numpy())

        f1 = f1_score(gts, preds, average='macro')
        print(f"Inception Epoch {epoch+1} | F1: {f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model_inc.state_dict(), "best_inc.pth")

    print("✅ Inception Done")

train_inception()

Inception Epoch 1 | F1: 0.8730
Inception Epoch 2 | F1: 0.8688
Inception Epoch 3 | F1: 0.8075
Inception Epoch 4 | F1: 0.8002
Inception Epoch 5 | F1: 0.8838
Inception Epoch 6 | F1: 0.8411
Inception Epoch 7 | F1: 0.8868
Inception Epoch 8 | F1: 0.8919
Inception Epoch 9 | F1: 0.8985
Inception Epoch 10 | F1: 0.9020
Inception Epoch 11 | F1: 0.9136
Inception Epoch 12 | F1: 0.9053
Inception Epoch 13 | F1: 0.9178
Inception Epoch 14 | F1: 0.9233
Inception Epoch 15 | F1: 0.8961
Inception Epoch 16 | F1: 0.9109
Inception Epoch 17 | F1: 0.8997
Inception Epoch 18 | F1: 0.9262
Inception Epoch 19 | F1: 0.9263
Inception Epoch 20 | F1: 0.9387
Inception Epoch 21 | F1: 0.9332
Inception Epoch 22 | F1: 0.9384
Inception Epoch 23 | F1: 0.9311
Inception Epoch 24 | F1: 0.9378
Inception Epoch 25 | F1: 0.9407
Inception Epoch 27 | F1: 0.9408
Inception Epoch 28 | F1: 0.9457
Inception Epoch 29 | F1: 0.9474
Inception Epoch 30 | F1: 0.9487
✅ Inception Done


In [14]:
model_eff.load_state_dict(torch.load("best_eff.pth", map_location=device))
model_inc.load_state_dict(torch.load("best_inc.pth", map_location=device))

model_eff.eval()
model_inc.eval()

InceptionV3(
  (Conv2d_1a_3x3): ConvNormAct(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2a_3x3): ConvNormAct(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Conv2d_2b_3x3): ConvNormAct(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
  )
  (Pool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): ConvNormAct(
    (conv): Conv2d(64, 80, kernel_size

In [16]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
import numpy as np

# ==============================
# TTA TRANSFORMS
# ==============================
def tta_transforms(images):

    # 1️⃣ Original
    t1 = images

    # 2️⃣ Horizontal flip
    t2 = torch.flip(images, dims=[3])

    # 3️⃣ Slight brightness increase
    t3 = torch.clamp(images * 1.03, 0, 1)

    # 4️⃣ Slight brightness decrease
    t4 = torch.clamp(images * 0.97, 0, 1)

    return [t1, t2, t3, t4]


# ==============================
# ENSEMBLE WEIGHTS
# ==============================
w_eff = 0.7
w_inc = 0.3

# TTA boost weights
boosts = [1, 1, 1.5, 1.5]

# ==============================
# EVALUATION
# ==============================
model_eff.eval()
model_inc.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Apply TTA
        tta_imgs = tta_transforms(images)

        # Store predictions
        final_probs = None

        # ==========================
        # TTA LOOP
        # ==========================
        for idx, tta_img in enumerate(tta_imgs):

            # EfficientNet prediction
            out_eff = model_eff(tta_img)

            # Inception prediction
            out_inc = model_inc(tta_img)

            # If inception returns auxiliary outputs
            if isinstance(out_inc, tuple):
                out_inc = out_inc[0]

            # Softmax probabilities
            prob_eff = torch.softmax(out_eff, dim=1)
            prob_inc = torch.softmax(out_inc, dim=1)

            # Weighted ensemble
            ensemble_prob = (
                w_eff * prob_eff +
                w_inc * prob_inc
            )

            # Apply TTA boost
            ensemble_prob = boosts[idx] * ensemble_prob

            # Accumulate
            if final_probs is None:
                final_probs = ensemble_prob
            else:
                final_probs += ensemble_prob

        # Average TTA predictions
        final_probs = final_probs / sum(boosts)

        # Final prediction
        preds = torch.argmax(final_probs, dim=1)

        # Store
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


# ==============================
# METRICS
# ==============================
accuracy = accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average='macro')

print(f"\n✅ Accuracy  : {accuracy:.4f}")
print(f"✅ Macro F1  : {macro_f1:.4f}")

# Optional detailed report
print("\n📊 Classification Report:\n")
print(classification_report(all_labels, all_preds))


✅ Accuracy  : 0.9519
✅ Macro F1  : 0.9520

📊 Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       405
           1       1.00      0.99      0.99       405
           2       0.92      0.90      0.91       405
           3       0.90      0.92      0.91       405

    accuracy                           0.95      1620
   macro avg       0.95      0.95      0.95      1620
weighted avg       0.95      0.95      0.95      1620



In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# ==============================
# CONFUSION MATRIX
# ==============================

cm = confusion_matrix(all_labels, all_preds)

print("Confusion Matrix:\n")
print(cm)

In [ ]:
# Class names
class_names = test_dataset.classes   # automatically gets folder names

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
!pip install grad-cam

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [ ]:
target_layers = [model_eff.features[-1]]

In [ ]:
target_layers = [model_inc.Mixed_7c]

In [ ]:
# Get one batch
images, labels = next(iter(test_loader))

image = images[0].unsqueeze(0).to(device)
label = labels[0]

print("True Label:", label)

In [ ]:
cam = GradCAM(
    model=model_eff,
    target_layers=target_layers
)

In [ ]:
# Target class
targets = [ClassifierOutputTarget(int(label))]

# Generate grayscale CAM
grayscale_cam = cam(
    input_tensor=image,
    targets=targets
)

grayscale_cam = grayscale_cam[0]

In [ ]:
# Target class
targets = [ClassifierOutputTarget(int(label))]

# Generate grayscale CAM
grayscale_cam = cam(
    input_tensor=image,
    targets=targets
)

grayscale_cam = grayscale_cam[0]

In [ ]:
visualization = show_cam_on_image(
    rgb_img,
    grayscale_cam,
    use_rgb=True
)

In [ ]:
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(visualization)
plt.title("GradCAM")
plt.axis('off')

plt.show()

In [ ]:
!pip install torchray

In [ ]:
from torchray.attribution.rise import rise

In [ ]:
images, labels = next(iter(test_loader))

image = images[0].unsqueeze(0).to(device)
label = labels[0]

In [ ]:
saliency = rise(
    model_eff,
    image,
    target=label
)

In [ ]:
saliency_map = saliency.squeeze().cpu().numpy()

In [ ]:
rgb_img = image[0].cpu().permute(1,2,0).numpy()

rgb_img = (rgb_img - rgb_img.min()) / (rgb_img.max() - rgb_img.min())

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(saliency_map, cmap='jet')
plt.title("RISE Saliency")
plt.axis('off')

plt.show()